# Trabajo Práctico 2 - Grupo 02

### Modelo XGBoost - Dataset Aumentado

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es evaluar si el data augmentation mejora el F1-macro de XGBoost.
Se usa `train_augmented_eda_balanced.csv` (67.000 filas, distribución ~33/33/33)
en lugar del train original (51.000 filas, distribución 40/20/40).

La hipótesis es que balancear la clase neutra reduce el sesgo del modelo hacia
las clases mayoritarias y mejora el F1 de la clase 1.

**Importante:** los resultados se comparan contra el mejor XGBoost sin augmentation
(0.67792 en Kaggle) para verificar si el experimento funcionó.

## Importación e instalación de dependencias

In [1]:
!pip install -q spacy
!python -m spacy download es_core_news_sm
!pip install xgboost

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     -------------- ------------------------- 4.7/12.9 MB 62.5 MB/s eta 0:00:01
     ---------------------------------------- 12.9/12.9 MB 63.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight

In [3]:
import sys
sys.path.insert(0, "../../../../")  # sube cuatro niveles: v6 → XGBoost → Entregas → TP2

In [4]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [5]:
# Dataset aumentado con reseñas sintéticas generadas por LLM
# Shape: 67.000 filas | Distribución: ~33% negativa / ~33% neutra / ~33% positiva
train = pd.read_csv("../../../../data/train_augmented_eda_balanced.csv")
test  = pd.read_csv("../../../../data/test.csv")

print(f"Train aumentado: {len(train):,} filas")
print(f"Distribucion:")
print(train['label'].value_counts().sort_index())
print(f"\nProporciones:")
print(train['label'].value_counts(normalize=True).sort_index().round(3))

Train aumentado: 67,000 filas
Distribucion:
label
0    22400
1    22200
2    22400
Name: count, dtype: int64

Proporciones:
label
0    0.334
1    0.331
2    0.334
Name: proportion, dtype: float64


In [7]:
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

print("Preprocesando...")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando...
Train: 53,600 | Val: 13,400 | Test: 8,500


## Entrega N6.1: XGBoost + TF-IDF baseline (dataset aumentado)

Baseline con los hiperparametros ganadores del mejor experimento sin augmentation
(`max_depth=7`, `learning_rate=0.2`, `colsample_bytree=1.0`, `min_child_weight=1`,
`n_estimators=700`, `subsample=0.8`, `reg_lambda=2`).

Permite ver si el dataset aumentado por si solo mejora el resultado antes de
hacer busqueda de hiperparametros. Con distribucion balanceada el desbalance
de clases ya no es el problema principal.

In [8]:
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("xgb", XGBClassifier(
        n_estimators=700,
        max_depth=7,
        learning_rate=0.2,
        colsample_bytree=1.0,
        min_child_weight=1,
        subsample=0.8,
        reg_lambda=2,
        reg_alpha=0,
        random_state=SEED, n_jobs=-1, eval_metric="mlogloss"
    ))])

weights_train = compute_sample_weight("balanced", y_train)
pipe.fit(X_train, y_train, xgb__sample_weight=weights_train)
y_pred = pipe.predict(X_val)
evaluate("xgb_tfidf_aug_baseline_v6_1", y_val, y_pred,
         hyperparams={"n_estimators": 700, "vectorizer": "TF-IDF",
                      "dataset": "augmented_67k", "class_weight": "balanced"})


=== xgb_tfidf_aug_baseline_v6_1 ===
Hiperparámetros: {'n_estimators': 700, 'vectorizer': 'TF-IDF', 'dataset': 'augmented_67k', 'class_weight': 'balanced'}

F1-macro:  0.7487
Precision: 0.7493
Recall:    0.7482
Accuracy:  0.7484

              precision    recall  f1-score   support

    negativa     0.7648    0.7665    0.7657      4480
      neutra     0.6753    0.6932    0.6842      4440
    positiva     0.8079    0.7848    0.7962      4480

    accuracy                         0.7484     13400
   macro avg     0.7493    0.7482    0.7487     13400
weighted avg     0.7496    0.7484    0.7489     13400

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3434     829       217
neutra         743    3078       619
positiva       313     651      3516


In [9]:
y_full = np.concatenate([y_train, y_val])
weights_full = compute_sample_weight("balanced", y_full)
pipe.fit(np.concatenate([X_train, X_val]), y_full, xgb__sample_weight=weights_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [10]:
save_model(pipe, "xgb_tfidf_aug_baseline_v6_1")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}),
                "submission_xgb_tfidf_aug_baseline_v6_1.csv");

Modelo guardado → models\xgb_tfidf_aug_baseline_v6_1.joblib
Predicción guardada → submissions\submission_xgb_tfidf_aug_baseline_v6_1.csv  (8500 predicciones)
Distribución: clase 0: 39.4%, clase 1: 22.2%, clase 2: 38.4%


## Entrega N6.2: XGBoost + TF-IDF + RandomizedSearch (dataset aumentado)

Busqueda de hiperparametros sobre el dataset aumentado. Los parametros
`ngram_range`, `colsample_bytree`, `max_depth`, `learning_rate` y `min_child_weight`
estan fijados por resultados de experimentos anteriores sin augmentation.

Se exploran `n_estimators`, `subsample`, `reg_alpha` y `reg_lambda` - los que
mostraron mas variacion en experimentos anteriores.

In [11]:
pipe = Pipeline([
    ("tfidf", make_tfidf(ngram_range=(1, 2), sublinear_tf=True)),
    ("xgb", XGBClassifier(
        max_depth=7,
        learning_rate=0.2,
        colsample_bytree=1.0,
        min_child_weight=1,
        random_state=SEED, n_jobs=-1, eval_metric="mlogloss"
    ))])

param_dist = {
    "xgb__n_estimators": [400, 500, 600, 700],
    "xgb__subsample":    [0.65, 0.7, 0.75, 0.8],
    "xgb__reg_alpha":    [0, 0.01, 0.1],
    "xgb__reg_lambda":   [1, 1.5, 2],
}

rs = RandomizedSearchCV(
    pipe, param_dist,
    n_iter=40, cv=5, scoring="f1_macro",
    n_jobs=2, random_state=SEED, verbose=1
)

In [12]:
print("Buscando mejores hiperparametros XGBoost + TF-IDF (dataset aumentado)")
weights_train = compute_sample_weight("balanced", y_train)
rs.fit(X_train, y_train, xgb__sample_weight=weights_train)

print(f"\nMejores hiperparametros: {rs.best_params_}")
print(f"Mejor F1-macro en CV: {rs.best_score_:.4f}")

y_pred = rs.best_estimator_.predict(X_val)
evaluate("xgb_tfidf_aug_rs_v6_2", y_val, y_pred, hyperparams={
    **rs.best_params_,
    "vectorizer": "TF-IDF",
    "dataset": "augmented_67k"
})

Buscando mejores hiperparametros XGBoost + TF-IDF (dataset aumentado)
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Mejores hiperparametros: {'xgb__subsample': 0.8, 'xgb__reg_lambda': 2, 'xgb__reg_alpha': 0, 'xgb__n_estimators': 700}
Mejor F1-macro en CV: 0.7332

=== xgb_tfidf_aug_rs_v6_2 ===
Hiperparámetros: {'xgb__subsample': 0.8, 'xgb__reg_lambda': 2, 'xgb__reg_alpha': 0, 'xgb__n_estimators': 700, 'vectorizer': 'TF-IDF', 'dataset': 'augmented_67k'}

F1-macro:  0.7487
Precision: 0.7493
Recall:    0.7482
Accuracy:  0.7484

              precision    recall  f1-score   support

    negativa     0.7648    0.7665    0.7657      4480
      neutra     0.6753    0.6932    0.6842      4440
    positiva     0.8079    0.7848    0.7962      4480

    accuracy                         0.7484     13400
   macro avg     0.7493    0.7482    0.7487     13400
weighted avg     0.7496    0.7484    0.7489     13400

Matriz de confusión (filas=real, cols=predicho):
          negativa  neu

In [13]:
y_full = np.concatenate([y_train, y_val])
weights_full = compute_sample_weight("balanced", y_full)
best_pipe = rs.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), y_full, xgb__sample_weight=weights_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [14]:
save_model(best_pipe, "xgb_tfidf_aug_rs_v6_2")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}),
                "submission_xgb_tfidf_aug_rs_v6_2.csv");

Modelo guardado → models\xgb_tfidf_aug_rs_v6_2.joblib
Predicción guardada → submissions\submission_xgb_tfidf_aug_rs_v6_2.csv  (8500 predicciones)
Distribución: clase 0: 39.4%, clase 1: 22.2%, clase 2: 38.4%
